# Epi Info AI MATCH validation lab - proposed V0.1

Validate the bounded 1:1 paired-analysis candidate for `MATCH exposure outcome MATCHVAR=set_id`. Independent SciPy calculations are compared with the deployed Rust/WebAssembly V0.16 kernel. Passing validates this candidate and fixture; it does not establish legacy Epi Info parity or statistical approval.

## Method under review

V0.1 includes only complete two-record sets with exactly one case, one control, and binary nonmissing exposure. It reports every excluded set. For included pairs, `b` is case exposed/control unexposed and `c` is case unexposed/control exposed. The proposed matched odds ratio is `b/c`; McNemar probabilities depend only on `b` and `c`; the central conditional exact interval applies Clopper-Pearson limits to `p=b/(b+c)` and transforms them by `p/(1-p)`.

In [ ]:
import csv, hashlib, io, math, sys
from collections import defaultdict
import scipy
from scipy.stats import beta, binom, chi2
from pyodide.http import pyfetch
from js import WebAssembly, Uint8Array
fixture_response = await pyfetch('../../validation-fixtures/matched-pairs-contract-v0.1.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
data_response = await pyfetch('../../examples/matched-case-control/matched-pairs-hand-audit.csv')
data_response.raise_for_status(); data_bytes = await data_response.bytes()
assert hashlib.sha256(data_bytes).hexdigest() == fixture['dataset']['sha256']
records = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
assert len(records) == fixture['dataset']['records'] == 21
wasm_response = await pyfetch('../../epi2x2.wasm')
wasm_response.raise_for_status()
instance = await WebAssembly.instantiate(Uint8Array.new(await wasm_response.buffer()), {})
rust = instance.instance.exports
{'python': sys.version, 'scipy': scipy.__version__, 'dataset_sha256': fixture['dataset']['sha256'], 'status': fixture['status']}

In [ ]:
def derive_pairs(source):
    groups = defaultdict(list)
    for record in source:
        groups[record['set_id']].append(record)
    pairs = {'caseExposedControlUnexposed': 0, 'caseUnexposedControlExposed': 0, 'bothExposed': 0, 'neitherExposed': 0}
    excluded = {'records': 0, 'sets': 0, 'missingAnalysisValueSets': 0, 'invalidAnalysisValueSets': 0, 'invalidCaseControlCompositionSets': 0, 'unsupportedVariableRatioSets': 0}
    included_records = included_sets = 0
    for members in groups.values():
        reason = None
        if any(not member[field].strip() for member in members for field in ('set_id', 'outcome', 'exposure')):
            reason = 'missingAnalysisValueSets'
        elif any(member[field] not in ('0', '1') for member in members for field in ('outcome', 'exposure')):
            reason = 'invalidAnalysisValueSets'
        elif len(members) != 2:
            reason = 'unsupportedVariableRatioSets'
        else:
            cases = [member for member in members if member['outcome'] == '1']
            controls = [member for member in members if member['outcome'] == '0']
            if len(cases) != 1 or len(controls) != 1:
                reason = 'invalidCaseControlCompositionSets'
            else:
                included_sets += 1; included_records += 2
                orientation = cases[0]['exposure'] + controls[0]['exposure']
                name = {'10': 'caseExposedControlUnexposed', '01': 'caseUnexposedControlExposed', '11': 'bothExposed', '00': 'neitherExposed'}[orientation]
                pairs[name] += 1
        if reason:
            excluded['sets'] += 1; excluded['records'] += len(members)
            excluded[reason] = excluded.get(reason, 0) + 1
    pairs['discordant'] = pairs['caseExposedControlUnexposed'] + pairs['caseUnexposedControlExposed']
    pairs['concordant'] = pairs['bothExposed'] + pairs['neitherExposed']
    return {'included': {'records': included_records, 'sets': included_sets}, 'excluded': excluded, 'pairs': pairs}
derived = derive_pairs(records)
assert derived['included'] == fixture['expected']['included']
assert derived['excluded'] == fixture['expected']['excluded']
assert derived['pairs'] == fixture['expected']['pairs']
derived

In [ ]:
b = derived['pairs']['caseExposedControlUnexposed']; c = derived['pairs']['caseUnexposedControlExposed']; n = b + c
alpha = 1 - fixture['expected']['matchedOddsRatio']['confidenceLevel']
lower_p = float(beta.ppf(alpha / 2, b, c + 1)) if b else 0.0
upper_p = float(beta.ppf(1 - alpha / 2, b + 1, c)) if c else 1.0
result = {
    'matchedOddsRatio': b / c if c else math.inf,
    'conditionalExactCentral': {'lower': lower_p / (1 - lower_p), 'upper': upper_p / (1 - upper_p)},
    'mcnemarUncorrected': {'chiSquare': (b-c)**2/n, 'pValue': float(chi2.sf((b-c)**2/n, 1))},
    'mcnemarContinuityCorrected': {'chiSquare': max(abs(b-c)-1, 0)**2/n, 'pValue': float(chi2.sf(max(abs(b-c)-1, 0)**2/n, 1))},
    'exactTwoSidedPValue': min(1.0, 2 * float(binom.cdf(min(b, c), n, 0.5))),
    'exactTwoSidedMidPValue': min(1.0, 2 * (float(binom.cdf(min(b, c), n, 0.5)) - 0.5 * float(binom.pmf(min(b, c), n, 0.5)))),
}
expected = fixture['expected']; tolerance = 1e-12
assert math.isclose(result['matchedOddsRatio'], expected['matchedOddsRatio']['value'], abs_tol=tolerance, rel_tol=0)
for name in ('lower', 'upper'): assert math.isclose(result['conditionalExactCentral'][name], expected['matchedOddsRatio']['conditionalExactCentral'][name], abs_tol=tolerance, rel_tol=0)
for actual_name, expected_name in (('mcnemarUncorrected', 'uncorrected'), ('mcnemarContinuityCorrected', 'continuityCorrected')):
    for name in ('chiSquare', 'pValue'): assert math.isclose(result[actual_name][name], expected['mcnemar'][expected_name][name], abs_tol=tolerance, rel_tol=0)
assert math.isclose(result['exactTwoSidedPValue'], expected['mcnemar']['exactTwoSidedPValue'], abs_tol=tolerance, rel_tol=0)
assert math.isclose(result['exactTwoSidedMidPValue'], expected['mcnemar']['exactTwoSidedMidPValue'], abs_tol=tolerance, rel_tol=0)
result

In [ ]:
candidate = {
    'matchedOddsRatio': float(rust.matched_odds_ratio(b, c)),
    'conditionalExactCentral': {'lower': float(rust.matched_odds_ratio_exact_lower(b, c, .95)), 'upper': float(rust.matched_odds_ratio_exact_upper(b, c, .95))},
    'mcnemarUncorrected': {'chiSquare': float(rust.matched_mcnemar_uncorrected(b, c))},
    'mcnemarContinuityCorrected': {'chiSquare': float(rust.matched_mcnemar_corrected(b, c))},
    'exactTwoSidedPValue': float(rust.matched_exact_two_sided(b, c)),
    'exactTwoSidedMidPValue': float(rust.matched_exact_mid_p_two_sided(b, c)),
}
candidate['mcnemarUncorrected']['pValue'] = float(rust.chi_square_p_value(candidate['mcnemarUncorrected']['chiSquare']))
candidate['mcnemarContinuityCorrected']['pValue'] = float(rust.chi_square_p_value(candidate['mcnemarContinuityCorrected']['chiSquare']))
assert math.isclose(candidate['matchedOddsRatio'], result['matchedOddsRatio'], abs_tol=tolerance, rel_tol=0)
for name in ('lower', 'upper'): assert math.isclose(candidate['conditionalExactCentral'][name], result['conditionalExactCentral'][name], abs_tol=tolerance, rel_tol=0)
for method in ('mcnemarUncorrected', 'mcnemarContinuityCorrected'):
    for name in ('chiSquare', 'pValue'): assert math.isclose(candidate[method][name], result[method][name], abs_tol=tolerance, rel_tol=0)
assert math.isclose(candidate['exactTwoSidedPValue'], result['exactTwoSidedPValue'], abs_tol=tolerance, rel_tol=0)
assert math.isclose(candidate['exactTwoSidedMidPValue'], result['exactTwoSidedMidPValue'], abs_tol=tolerance, rel_tol=0)
candidate

In [ ]:
def odds_state(b, c):
    if b == 0 and c == 0: return {'state': 'unavailable', 'value': None}
    if c == 0: return {'state': 'positive-infinity', 'value': None}
    if b == 0: return {'state': 'zero', 'value': 0.0}
    return {'state': 'finite', 'value': b/c}
boundary = {'no_discordance': odds_state(0, 0), 'zero': odds_state(0, 2), 'infinity': odds_state(2, 0)}
assert boundary == {'no_discordance': {'state': 'unavailable', 'value': None}, 'zero': {'state': 'zero', 'value': 0.0}, 'infinity': {'state': 'positive-infinity', 'value': None}}
boundary

In [ ]:
assert derive_pairs(list(reversed(records))) == derived  # row-order invariance
relabeled = [{**record, 'set_id': 'RELABELED-' + record['set_id']} for record in records]
assert derive_pairs(relabeled) == derived  # matched-set label invariance
reversed_exposure = [{**record, 'exposure': ('0' if record['exposure'] == '1' else '1' if record['exposure'] == '0' else record['exposure'])} for record in records]
reversed_result = derive_pairs(reversed_exposure)
assert reversed_result['pairs']['caseExposedControlUnexposed'] == c
assert reversed_result['pairs']['caseUnexposedControlExposed'] == b
assert math.isclose((c/b), 1/result['matchedOddsRatio'], abs_tol=1e-15, rel_tol=0)
reverse_lower_p = float(beta.ppf(alpha/2, c, b+1)); reverse_upper_p = float(beta.ppf(1-alpha/2, c+1, b))
assert math.isclose(reverse_lower_p/(1-reverse_lower_p), 1/result['conditionalExactCentral']['upper'], abs_tol=1e-12, rel_tol=0)
assert math.isclose(reverse_upper_p/(1-reverse_upper_p), 1/result['conditionalExactCentral']['lower'], abs_tol=1e-12, rel_tol=0)
assert math.isclose(2*float(binom.cdf(min(c,b), n, .5)), result['exactTwoSidedPValue'], abs_tol=1e-15, rel_tol=0)
print('PASS: row-order invariance, set-label invariance, and exposure-reversal reciprocity')

## Result and next gate

A clean run establishes agreement between the deployed Rust/WebAssembly V0.16 candidate and the independent SciPy reproduction. The Program Editor now executes this same candidate through the typed browser adapter and cancellable Worker for the bounded 1:1 form. Experienced field-user comparison with historical Epi Info workflows remains required for any legacy-parity claim.